# YodaX readiness audit — October 25

Before trying another model, I want to check the evidence we already have. This notebook reads the saved two-stock study; it does not retrain anything or write to the live database. Passing date checks would not establish that global events cause price movements.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "research_pipeline").exists():
    root = root.parent
sys.path.insert(0, str(root))
from research_pipeline.readiness_audit import audit, markdown
from IPython.display import Markdown, display
result = audit(root)
display(Markdown(markdown(result)))

# YodaX readiness audit — October 25, 2026

This checks the saved pilot datasets and evaluation splits. It does not train a new model or change the live candidate.

## Findings

- Feature rows: 980; settled rows: 970; complete exposure windows: 828.
- Event archive days: 641; missing dates within that interval: 18.
- All models use identical stock/date evaluation keys: True.
- Duplicate model/stock/date predictions: 0.
- Frozen shadow artifact checksums match: True.

| Stock | Price rows | Missing sessions | Duplicate dates | Invalid closes |
|---|---:|---:|---:|---:|
| AAPL | 745 | 0 | 0 | 0 |
| FCX | 745 | 0 | 0 | 0 |

## Chronological splits

| Test interval (end exclusive) | Train rows | Test rows / dates | Last training outcome | Checks |
|---|---:|---:|---|---|
| 2025-09-01 → 2026-01-01 | 244 | 160 / 80 | 2025-08-29 | PASS |
| 2026-01-01 → 2026-07-01 | 414 | 236 / 118 | 2025-12-31 | PASS |

Checks cover target-date purging, disjoint asset/date keys and settled labels. Preprocessing is fitted inside each training fold in the implementation; this report does not independently reconstruct every historical fit.

## Open issues

- The 2015 ambition is not the coverage of this event-aware pilot: event archives start in 2025.
- July 2026 onward was examined previously; it is not an untouched final test set.
- Country presence is a proxy for exposure. Event-to-company relevance and causal effects have not been validated.
- Chronological checks do not establish original source availability: event and disclosure delays are conservative assumptions.
- A fixed two-stock universe has selection bias; no conclusion about all stocks or commodities follows.
- No independent vendor reconciliation, drift clearance, profitability evaluation or calibrated prediction intervals are established.
- Historical comparison results were already inspected. New criteria are prospective, not preregistration of those results.

Missing event archive dates: 2025-06-14, 2025-06-15, 2025-06-16, 2025-06-17, 2025-06-18, 2025-06-19, 2025-06-20, 2025-06-21, 2025-06-22, 2025-06-23, 2025-06-24, 2025-06-25, 2025-06-26, 2025-06-27, 2025-06-28, 2025-06-29, 2025-06-30, 2025-07-01.

Exposure age-limit dates: AAPL: 2026-11-04; FCX: 2027-03-01.

## Decision

**Continue private shadow evaluation; do not promote.** The mechanical split checks pass, but relevance review, source freshness and prospective performance evidence remain open. There is no defensible single dataset-quality score.

The accompanying JSON contains input checksums, per-feature missingness, coverage bounds and individual checks. Reproduce with `.venv-research/bin/python -m research_pipeline.readiness_audit`.


## What needs attention next

The split checks tell us about dates, not business relevance. Next I would review sampled event links against their sources and each company’s exposure. A country mention can be accurate and still be a poor predictor for Apple or Freeport-McMoRan.

Missing archives also affect which periods we evaluate. Excluding incomplete windows avoids false “no events” signals, but we still need to investigate the resulting selection bias.

In [2]:
import pandas as pd
display(pd.DataFrame(result["splits"])[["start", "end_exclusive", "train_rows", "test_rows", "test_dates", "last_training_target"]])
display(pd.Series(result["quality"]["feature_missing_fraction"], name="missing_fraction").sort_values(ascending=False).to_frame())
assert all(all(split["checks"].values()) for split in result["splits"])
assert result["comparison_asset_dates_identical"]
assert result["prediction_duplicate_keys"] == 0
assert all(result["frozen_artifacts_match"].values())

,start,end_exclusive,train_rows,test_rows,test_dates,last_training_target
0,2025-09-01,2026-01-01,244,160,80,2025-08-29
1,2026-01-01,2026-07-01,414,236,118,2025-12-31


,missing_fraction
exposed_negotiation_halt_share_7d,0.155102
exposed_cooperation_share_7d,0.155102
exposed_incoming_sanctions_share_7d,0.155102
exposed_sanctions_share_7d,0.155102
exposed_conflict_share_7d,0.155102
exposed_event_log_count_7d,0.155102
material_conflict_share_7d,0.120408
event_log_count_7d,0.120408
material_cooperation_share_7d,0.120408
exposure_age_days,0.078571


## Decision

Continue private shadow evaluation. Do not promote yet. The [experiment plan](../docs/experiment-plan.md) records the comparison rules, milestones and promotion gates. Early December is a progress review: the current uncertainty method requires at least 60 matched cutoff dates, so six weeks alone is not enough.

The later historical periods have already been examined and are not an untouched holdout. This audit uses the October 23 study snapshots; it is not a fresh download. The machine-readable report in `docs/audits/2026-10-25/readiness.json` records input hashes, coverage and individual checks.